# Pixels to Predictions

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q "transformers>=4.45.0,<5.0.0" "accelerate>=0.27.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" "datasets" "pillow"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 209.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.1 MB/s eta 0:00:00


In [3]:
import os, gc, re, json, warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import pandas as pd
from PIL import Image
from datasets import Dataset
import random
import numpy as np
from torch.utils.data import DataLoader
from torch.amp import autocast
import torchvision.transforms as T

from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import bitsandbytes as bnb
from tqdm.auto import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


Device: cuda


In [17]:
CFG = dict(
    data_root      = "/content/drive/MyDrive/dl_finals",
    out_dir        = "/content/drive/MyDrive/dl_finals/checkpoints/smolvlm-weights-final-run",
    prev_check_dir = "/content/drive/MyDrive/dl_finals/checkpoints/smolvlm-weights-final",
    submission     = "/content/drive/MyDrive/dl_finals/submission.csv",

    model_name  = "HuggingFaceTB/SmolVLM-500M-Instruct",

    use_nf4      = True,
    double_quant = True,

    lora_r       = 4,
    lora_alpha   = 8,
    lora_dropout = 0.05,
    use_dora     = True,
    lora_targets = ["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],

    # epochs         = 22, # 22 Epoch run, each epoch takes 25 mins
    epochs         = 3, # last 3 epoch run from last checkpoint
    train_bs       = 2,
    grad_accum     = 16,
    lr             = 2e-4,
    warmup_ratio   = 0.05,
    max_grad_norm  = 1.0,
    patience       = 6,
    eval_every     = 100,

    train_res  = 256,
    infer_res  = 384,

    infer_bs       = 8,
    max_new_tokens = 50,

    num_workers = 2,
    seed        = 42,
)

torch.manual_seed(CFG["seed"])
os.makedirs(CFG["out_dir"], exist_ok=True)


In [5]:
def flush_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


In [8]:
root = CFG["data_root"]
train_df = pd.read_csv(f"{root}/train.csv")
val_df   = pd.read_csv(f"{root}/val.csv")
test_df  = pd.read_csv(f"{root}/test.csv")

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print("Columns:", train_df.columns.tolist())

Train: 3,109  |  Val: 1,048  |  Test: 1,008
Columns: ['id', 'image_path', 'question', 'choices', 'num_choices', 'answer', 'hint', 'lecture', 'solution', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


In [9]:
print(f"Test rows: {len(test_df):,}")
print(test_df.head(3))
test_df = pd.read_csv("/content/drive/MyDrive/dl_finals/test.csv")

Test rows: 1,008
           id                  image_path  \
0  test_01750  images/test/test_01750.png   
1  test_00128  images/test/test_00128.png   
2  test_02891  images/test/test_02891.png   

                                            question  \
0  Based on clues in the text, why would farmers ...   
1  What is the probability that an American curl ...   
2  What is the expected ratio of offspring with a...   

                                             choices  num_choices  \
0  ["The cats were thought to be visiting goddess...            4   
1                ["0/4", "2/4", "4/4", "3/4", "1/4"]            5   
2                ["4:0", "3:1", "1:3", "0:4", "2:2"]            5   

                                                hint  \
0  Read the text about cats.\nCats are among the ...   
1  In a group of American curl cats, some individ...   
2  This passage describes the ground spot color t...   

                                             lecture           task   grade

In [10]:
COD_SYSTEM = "Reason step by step. Each step: <=5 words. End with #### <answer_index>."

def _safe(row, col):
    v = row.get(col, "")
    return str(v).strip() if (v and str(v).strip() not in ("", "nan")) else ""

def build_messages(row, include_answer=True):
    choices = json.loads(row["choices"])
    choices_text = "\n".join(f"{i}. {c}" for i, c in enumerate(choices))

    meta_lines = []
    for col in ("subject", "grade", "topic", "skill"):
        val = _safe(row, col)
        if val:
            meta_lines.append(f"{col.capitalize()}: {val}")
    meta_block = ("\n".join(meta_lines) + "\n\n") if meta_lines else ""

    hint_block    = f"Hint: {_safe(row, 'hint')}\n" if _safe(row, 'hint') else ""
    lecture_block = f"Context: {_safe(row, 'lecture')}\n" if _safe(row, 'lecture') else ""

    user_text = (
        f"{meta_block}"
        f"{COD_SYSTEM}\n\n"
        f"Science multiple-choice question:\n"
        f"Q: {row['question']}\n\n"
        f"Choices:\n{choices_text}\n\n"
        f"{hint_block}"
        f"{lecture_block}"
        f"Answer index (0-{len(choices)-1}):"
    )

    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}
    ]

    if include_answer and "answer" in row:
        ans_idx = int(row["answer"])
        messages.append(
            {"role": "assistant", "content": [{"type": "text", "text": f"Visual confirms option {ans_idx}. #### {ans_idx}"}]}
        )

    return messages

In [11]:
def extract_answer(text: str, num_choices: int) -> int:
    def _valid(d):
        return 0 <= int(d) < num_choices

    m = re.search(r'####\s*(\d)', text)
    if m and _valid(m.group(1)):
        return int(m.group(1))

    m = re.search(r'(?:the\s+)?(?:correct\s+)?answer\s*(?:is|:|-)?\s*(\d)', text, re.IGNORECASE)
    if m and _valid(m.group(1)):
        return int(m.group(1))

    m = re.search(r'(?:option|choice)\s*(\d)', text, re.IGNORECASE)
    if m and _valid(m.group(1)):
        return int(m.group(1))

    for d in reversed(re.findall(r'\b(\d)\b', text)):
        if _valid(d):
            return int(d)

    return -1

In [12]:
def build_dataset(df, split="train"):
    data = []
    include_ans = (split != "test")
    for _, row in df.iterrows():
        img_path = f"{root}/images/{row['image_path']}"
        if not os.path.exists(img_path):
            img_path = f"{root}/{row['image_path']}"
        if not os.path.exists(img_path): # In Case there is no image go on
            continue
        item = {
            "id":          row["id"],
            "image":       img_path,
            "messages":    build_messages(row, include_answer=include_ans),
            "num_choices": int(row["num_choices"]),
        }
        if include_ans:
            item["answer"] = str(int(row["answer"]))
        data.append(item)
    print(f"  {split}: {len(data):,} examples ({len(df)-len(data)} skipped)")
    return Dataset.from_list(data)

train_dataset = build_dataset(train_df, "train")
val_dataset   = build_dataset(val_df,   "val")
test_dataset  = build_dataset(test_df,  "test")

  train: 3,109 examples (0 skipped)
  val: 1,048 examples (0 skipped)
  test: 1,008 examples (0 skipped)


In [13]:
processor = AutoProcessor.from_pretrained(CFG["model_name"])
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"
print("Processor loaded.")

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Processor loaded.


In [14]:
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit              = CFG["use_nf4"],
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = CFG["double_quant"],
    bnb_4bit_compute_dtype    = torch.bfloat16,
) if CFG["use_nf4"] else None

flush_memory()


model = AutoModelForVision2Seq.from_pretrained(
    CFG["model_name"],
    quantization_config = bnb_cfg,
    device_map          = "auto",
    dtype               = torch.bfloat16,
    attn_implementation = "eager",
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [16]:
lora_cfg = LoraConfig(
    r              = CFG["lora_r"],
    lora_alpha     = CFG["lora_alpha"],
    target_modules = CFG["lora_targets"],
    lora_dropout   = CFG["lora_dropout"],
    bias           = "none",
    task_type      = "CAUSAL_LM",
    use_dora       = CFG["use_dora"],
)

model = PeftModel.from_pretrained(
    model,
    CFG["prev_check_dir"],
    is_trainable=True
)

model.print_trainable_parameters()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert n_trainable <= 5_000_000, f"Over budget! {n_trainable:,} trainable params."

trainable params: 2,162,688 || all params: 509,644,992 || trainable%: 0.4244


In [18]:
train_transform = T.Compose([
    T.RandomResizedCrop(CFG["train_res"], scale=(0.85, 1.0)),
    T.RandomRotation(degrees=5),
    T.ColorJitter(brightness=0.2, contrast=0.1),
])

infer_transform = T.Compose([
    T.Resize((CFG["infer_res"], CFG["infer_res"])),
])

def collate_fn(batch, training=True):
    is_train  = "answer" in batch[0]
    transform = train_transform if (is_train and training) else infer_transform

    images, messages = [], []
    for x in batch:
        images.append(transform(Image.open(x["image"]).convert("RGB")))
        messages.append(x["messages"])

    texts = [processor.apply_chat_template(msg, add_generation_prompt=not is_train) for msg in messages]

    inputs = processor(
        text=texts,
        images=[[img] for img in images],
        padding=True,
        return_tensors="pt",
    )

    if is_train:
        labels = inputs["input_ids"].clone()
        labels[inputs["attention_mask"] == 0] = -100

        prompt_texts = [processor.apply_chat_template([msg[0]], add_generation_prompt=True) for msg in messages]
        prompt_inputs = processor(
            text=prompt_texts,
            images=[[img] for img in images],
            padding=True,
            return_tensors="pt",
        )
        for i, plen in enumerate(prompt_inputs["attention_mask"].sum(dim=1).tolist()):
            labels[i, :plen] = -100

        inputs["labels"] = labels

    metadata = [{"id": x["id"], "answer": x.get("answer"), "num_choices": x["num_choices"]} for x in batch]
    return inputs, metadata

def train_collate(b): return collate_fn(b, training=True)
def eval_collate(b):  return collate_fn(b, training=False)

In [19]:
train_loader = DataLoader(
    train_dataset, batch_size=CFG["train_bs"], shuffle=True,
    collate_fn=train_collate, num_workers=CFG["num_workers"], pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG["train_bs"], shuffle=False,
    collate_fn=train_collate, num_workers=CFG["num_workers"], pin_memory=True,
)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")

Train batches: 1,555
Val batches: 524


In [20]:
optimizer = bnb.optim.PagedAdamW8bit(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["lr"], betas=(0.9, 0.999), weight_decay=0.01,
)

total_opt_steps = (len(train_loader) // CFG["grad_accum"]) * CFG["epochs"]
warmup_steps    = max(1, int(CFG["warmup_ratio"] * total_opt_steps))

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_opt_steps,
)

print(f"Optimizer: PagedAdamW8bit  lr={CFG['lr']}")
print(f"Total opt steps: {total_opt_steps}  |  warmup: {warmup_steps}")

Optimizer: PagedAdamW8bit  lr=0.0002
Total opt steps: 291  |  warmup: 14


In [21]:
@torch.no_grad()
def run_val_loss():
    model.eval()
    total_loss = 0.0
    for batch, _ in val_loader:
        batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
        with autocast("cuda", dtype=torch.bfloat16):
            total_loss += model(**batch).loss.item()
    model.train()
    return total_loss / len(val_loader)

In [22]:
best_val_loss     = float("inf")
epochs_no_improve = 0
global_step       = 0
micro_step        = 0

model.train()

for epoch in range(CFG["epochs"]):
    epoch_loss = 0.0
    accum_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']}")
    for step, (batch, _) in enumerate(pbar):
        batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}

        with autocast("cuda", dtype=torch.bfloat16):
            out  = model(**batch)
            loss = out.loss / CFG["grad_accum"]

        loss.backward()
        micro_step += 1
        epoch_loss += out.loss.item()
        accum_loss += out.loss.item()

        if micro_step % CFG["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                CFG["max_grad_norm"]
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % CFG["eval_every"] == 0:
                avg = accum_loss / (CFG["grad_accum"] * CFG["eval_every"])
                pbar.set_postfix({"loss": f"{avg:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
                accum_loss = 0.0

    val_loss  = run_val_loss()
    avg_train = epoch_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} | train_loss: {avg_train:.4f} | val_loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss     = val_loss
        epochs_no_improve = 0
        model.save_pretrained(CFG["out_dir"])
        processor.save_pretrained(CFG["out_dir"])
        print(f"  New best ({val_loss:.4f}) - checkpoint saved")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{CFG['patience']})")
        if epochs_no_improve >= CFG["patience"]:
            print(f"  Early stopping at epoch {epoch+1}.")
            break

    flush_memory()

print(f"\nTraining done. Best val_loss: {best_val_loss:.4f}")

Epoch 1/3:   0%|          | 0/1555 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...



Epoch 1 | train_loss: 0.0749 | val_loss: 0.1104
  New best (0.1104) - checkpoint saved


Epoch 2/3:   0%|          | 0/1555 [00:00<?, ?it/s]


Epoch 2 | train_loss: 0.0639 | val_loss: 0.1123
  No improvement (1/6)


Epoch 3/3:   0%|          | 0/1555 [00:00<?, ?it/s]


Epoch 3 | train_loss: 0.0585 | val_loss: 0.1114
  No improvement (2/6)

Training done. Best val_loss: 0.1104


In [23]:
del model, optimizer, scheduler
flush_memory()

In [24]:
processor = AutoProcessor.from_pretrained(CFG["out_dir"])
processor.tokenizer.padding_side = "left"

base_for_infer = AutoModelForVision2Seq.from_pretrained(
    CFG["model_name"],
    quantization_config = bnb_cfg,
    device_map          = "auto",
    dtype               = torch.bfloat16,
    attn_implementation = "eager",
)

model = PeftModel.from_pretrained(base_for_infer, CFG["out_dir"])
model.eval()

flush_memory()

print("Best checkpoint loaded for inference.")

Best checkpoint loaded for inference.


In [25]:
test_loader = DataLoader(
    test_dataset, batch_size=CFG["infer_bs"], shuffle=False,
    collate_fn=eval_collate, num_workers=CFG["num_workers"], pin_memory=True,
)

preds, pred_ids = [], []

with torch.inference_mode():
    for batch, metadata in tqdm(test_loader, desc="Predicting"):
        batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
        batch.pop("labels", None)
        prompt_len = batch["input_ids"].shape[1]

        with autocast("cuda", dtype=torch.bfloat16):
            out = model.generate(
                **batch,
                max_new_tokens=CFG["max_new_tokens"],
                do_sample=False,
                use_cache=True,
            )

        for i, meta in enumerate(metadata):
            decoded   = processor.decode(out[i, prompt_len:], skip_special_tokens=True)
            n_choices = int(test_df[test_df["id"] == meta["id"]]["num_choices"].values[0])
            pred      = extract_answer(decoded, n_choices)
            preds.append(int(pred) if pred != -1 else 0)
            pred_ids.append(meta["id"])

submission = pd.DataFrame({"id": pred_ids, "answer": preds})
submission = test_df[["id"]].merge(submission, on="id", how="left")
submission["answer"] = submission["answer"].fillna(0).astype(int)

assert len(submission) == len(test_df), f"Row count mismatch: {len(submission)} vs {len(test_df)}"
assert submission["answer"].between(0, 9).all(), "Answer out of range"

submission.to_csv(CFG["submission"], index=False)
print(f"Submission saved -> {CFG['submission']}")
print(submission.head(10))

Predicting:   0%|          | 0/126 [00:00<?, ?it/s]

Submission saved -> /content/drive/MyDrive/dl_finals/submission.csv
           id  answer
0  test_01750       1
1  test_00128       1
2  test_02891       1
3  test_02425       0
4  test_00930       0
5  test_03725       0
6  test_00009       1
7  test_02880       1
8  test_01208       1
9  test_00619       1


In [26]:
val_preds, val_ids, val_targets = [], [], []

model.eval()

with torch.inference_mode():
    for batch, metadata in tqdm(val_loader, desc="Val Inference"):
        batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
        batch.pop("labels", None)
        prompt_len = batch["input_ids"].shape[1]

        with autocast("cuda", dtype=torch.bfloat16):
            out = model.generate(
                **batch,
                max_new_tokens = CFG["max_new_tokens"],
                do_sample      = False,
                use_cache      = True,
            )

        for i, meta in enumerate(metadata):
            decoded   = processor.decode(out[i, prompt_len:], skip_special_tokens=True)
            n_choices = int(val_df[val_df["id"] == meta["id"]]["num_choices"].values[0])
            pred      = extract_answer(decoded, n_choices)
            if pred == -1:
                pred = 0

            val_preds.append(int(pred))
            val_ids.append(meta["id"])

            gt = val_df[val_df["id"] == meta["id"]]["answer"].values
            val_targets.append(int(gt[0]))


results_df = pd.DataFrame({"id": val_ids, "pred": val_preds, "target": val_targets})
correct = (results_df["pred"] == results_df["target"]).sum()
total = len(results_df)
acc = correct / total
print(f"Val Accuracy: {correct}/{total} = {acc*100:.2f}%")

Val Inference:   0%|          | 0/524 [00:00<?, ?it/s]

Val Accuracy: 429/1048 = 40.94%
